# QF600 – Pairs trading in crypto: Gatev replication, clustering, and threshold tests

One notebook for the whole project. Three models, one engine, one data set, one in-sample (IS) / out-of-sample (OOS) split.

| Model | Pair selection | Trading strategy |
|---|---|---|
| **Model 1: Gatev baseline** | minimum distance (sum of squared deviations, SSD) across all coins | original 2σ (standard deviation, SD) rule |
| **Model 2: AP + SSD** | principal component analysis (PCA) + affinity propagation (AP) clusters, then minimum SSD within clusters | same 2σ rule |
| **Model 3: AP + SSD, modified** | same clustering as Model 2 | entry threshold chosen in-sample from a fixed grid (1.0, 1.5, 2.0, 2.5, 3.0 SD) |

**Approach: try to falsify, not to optimise.** The hypotheses, the threshold grid, the selection rule and the decision rules below were fixed *before* running this notebook. Every test is reported, including those that fail.

### Pre-registered hypotheses (one-sided tests, 5% level, Newey–West t > 1.65)
| | Hypothesis | Evidence that would falsify it |
|---|---|---|
| **H1** | Clustering improves on Gatev | Model 2 does not beat Model 1 OOS, **or** random clusters of the same sizes (placebo) do as well as AP |
| **H2** | A threshold chosen in-sample beats 2 SD out-of-sample | Model 3 (IS-chosen threshold) does not beat Model 2 at 2 SD OOS, after a 10 bp fee |
| **H3** | The selected pairs beat random pairs | ≥ 5% of random-pair bootstrap portfolios do as well OOS |
| **H4** | Profits survive realistic trading costs | OOS mean return at 10 bp per leg is not significantly above zero |

**Selection rule for Model 3 (fixed in advance):** the threshold with the highest in-sample Sharpe ratio for the Top 20 portfolio, committed capital, one-day wait, net of 10 bp per leg. That single choice is then scored once on OOS.

**Honesty note:** we had already seen OOS results for Models 1 and 2 at 2 SD before this notebook was written. The OOS period is therefore not perfectly untouched for those two models; it *is* untouched for the threshold grid and the placebo tests.

## 1. Settings

In [1]:
import os, glob, math, zipfile, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import AffinityPropagation
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from lets_plot import *
warnings.filterwarnings("ignore")
LetsPlot.setup_html()
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

# ---- data and sample --------------------------------------------------------------------------------
DATA_DIR   = "crypto_cache"      # folder (or full path) with one CSV per coin; crypto_cache.zip next to the notebook also works
START      = "2018-03-01"        # first day on which every coin in the universe has a price
EXCLUDE    = {"USDT"}            # stablecoin: no relative price movement to trade
IS_SHARE   = 0.75                # share of the data period that is in-sample

# ---- Gatev rules (Models 1 and 2) ------------------------------------------------------------------
FORM_M, TRADE_M = 12, 6          # 12-month formation, 6-month trading, new cohort every month
BASE_SD    = 2.0                 # Gatev entry threshold
PORTS      = {"Top 5": 5, "Top 20": 20, "All": None}
NW_LAGS    = 6

# ---- Model 2 clustering (qf627 Week 7, recipe B) ---------------------------------------------------
N_COMPONENTS = 5
PCA_SENSITIVITY = [3, 5, 10]

# ---- Model 3 and falsification tests (fixed before running) ----------------------------------------
THRESHOLDS     = [1.0, 1.5, 2.0, 2.5, 3.0]
SELECT_COST_BP = 10              # realistic fee used to choose the Model 3 threshold and to test H2 / H4
FEE_GRID_BP    = [0, 5, 10, 20, 30, 50, 75, 100]   # bp per leg per execution (a round trip = 4 leg executions)
N_BOOT         = 200             # random-pairs bootstrap runs (Gatev used 200)
N_PLACEBO      = 100             # random-cluster placebo runs
T_CRIT         = 1.65            # one-sided 5%
SEED           = 600

EXPORT_PNG = True                # also save every chart as a PNG in ./figures
BLUE, ORANGE, AQUA, YELLOW, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0"
MODEL_COL = {"Model 1": MUTED, "Model 2": BLUE, "Model 3": ORANGE}

def show(p, name):
    if EXPORT_PNG:
        os.makedirs("figures", exist_ok=True)
        try: ggsave(p, f"{name}.png", path="figures")
        except Exception as e: print(f"(PNG export of {name} skipped: {type(e).__name__})")
    p.show()

## 2. Load and clean the data

In [2]:
def find_data_dir(d):
    """The per-coin CSV folder: DATA_DIR, a crypto_cache folder nearby, or crypto_cache*.zip unzipped here."""
    if glob.glob(os.path.join(d, "*.csv")):
        return d
    for z in sorted(glob.glob("*crypto_cache*.zip")):
        with zipfile.ZipFile(z) as zf: zf.extractall(".")
        print(f"Unzipped {z}")
    for cand in sorted(glob.glob("**/crypto_cache", recursive=True)) + sorted(glob.glob("../*/crypto_cache")):
        if glob.glob(os.path.join(cand, "*.csv")):
            return cand
    raise FileNotFoundError(f"No coin CSVs in '{os.path.abspath(d)}'. Put the crypto_cache folder (or crypto_cache.zip) "
                            f"next to this notebook, or set DATA_DIR in section 1 to the folder's full path.")

SPIKE, REVERT = 4.0, 0.5   # one-day price ratio above 4x or below 1/4x is suspicious; back within +/-50% = spike

def clean_series(s, name, log):
    """Repair one-off bad ticks; set non-reverting jumps (level shifts) to missing."""
    s = s.copy()
    for _ in range(200):
        r = s.pct_change(fill_method=None)
        bad = r.index[(((1 + r) > SPIKE) | ((1 + r) < 1 / SPIKE)).fillna(False)]
        if len(bad) == 0:
            break
        i = s.index.get_loc(bad[0]); pre = s.iloc[i - 1]
        nxt = s.iloc[i + 1:i + 3]
        if len(nxt) and ((nxt / pre - 1).abs() < REVERT).any():
            j = i
            while j < len(s) and j < i + 3 and abs(s.iloc[j] / pre - 1) >= REVERT:
                s.iloc[j] = pre; j += 1
            log.append((name, bad[0].date(), "spike repaired", float(r.iloc[i])))
        else:
            s.iloc[i] = np.nan
            log.append((name, bad[0].date(), "level shift -> missing", float(r.iloc[i])))
    return s

DATA_DIR = find_data_dir(DATA_DIR)
close, vol, skipped, log = {}, {}, [], []
for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    base = os.path.basename(f)
    if base.endswith(".partial.csv"):
        skipped.append(base); continue
    d = pd.read_csv(f, parse_dates=["date"]).sort_values("date").drop_duplicates("date", keep="last").set_index("date")
    sym = str(d["symbol"].iloc[0]).upper()
    idx = pd.date_range(d.index.min(), d.index.max(), freq="D")
    c = d["close"].astype(float).reindex(idx); c[c <= 0] = np.nan
    close[sym] = clean_series(c, sym, log)
    vol[sym] = d["volumeto"].astype(float).reindex(idx)

first = {k: s.first_valid_index() for k, s in close.items()}
universe = sorted(k for k in close if k not in EXCLUDE and first[k] <= pd.Timestamp(START))
late     = sorted(k for k in close if k not in EXCLUDE and first[k] > pd.Timestamp(START))
last_common = min(close[k].last_valid_index() for k in universe)
END = last_common if last_common.is_month_end else last_common - pd.offsets.MonthEnd(1)

px  = pd.DataFrame(close)[universe].loc[START:END]
vlm = pd.DataFrame(vol)[universe].loc[START:END]
dates = px.index
P = px.to_numpy()
TRADEABLE = (np.isfinite(P) & (vlm.to_numpy() > 0))          # a "trade day": a price and non-zero USD volume

cl_log = pd.DataFrame(log, columns=["coin", "date", "action", "daily_return"])
print(f"Coin files read from: {os.path.abspath(DATA_DIR)}")
print(f"Skipped files: {skipped}; excluded: {sorted(EXCLUDE)}; listed after {START} (dropped): {late}")
print(f"Universe: {len(universe)} coins -> {', '.join(universe)}")
print(f"Daily data used: {dates[0].date()} to {dates[-1].date()} ({len(dates)} days)")
print("Cleaning actions inside the sample:", cl_log[(cl_log.date >= dates[0].date()) & (cl_log.date <= dates[-1].date())]
      .groupby("action").size().to_dict())

Coin files read from: /Users/personal/Desktop/local/asset_pricing_fnl/crypto_cache
Skipped files: ['VET.partial.csv']; excluded: ['USDT']; listed after 2018-03-01 (dropped): ['FCT', 'GLM']
Universe: 41 coins -> ADA, ARK, BCH, BLK, BNB, BTC, BTS, DASH, DCR, DGB, DOGE, ETC, ETH, KMD, LINK, LSK, LTC, MAID, NAV, NEO, NMC, NXT, OMNI, PIVX, PPC, QTUM, RDD, REP, SC, STEEM, STRAX, SYS, TRX, VTC, WAVES, XCP, XLM, XMR, XRP, XVG, ZEC
Daily data used: 2018-03-01 to 2026-09-30 (3136 days)
Cleaning actions inside the sample: {'level shift -> missing': 24, 'spike repaired': 13}


## 3. Sample and the in-sample / out-of-sample split

In [3]:
months = pd.period_range(dates[0], dates[-1], freq="M")
n_m = len(months)
mstart = np.append(np.searchsorted(dates.values, months.to_timestamp().values), len(dates))
n_is = int(round(IS_SHARE * n_m))
is_months, oos_months = months[:n_is], months[n_is:]
OOS0 = oos_months[0]
is_m  = lambda idx: idx < OOS0
oos_m = lambda idx: idx >= OOS0

def rng_str(ms): return f"{ms[0].strftime('%b %Y')} – {ms[-1].strftime('%b %Y')}"
display(pd.DataFrame([
    ["Full data period", rng_str(months), n_m, "formation + trading"],
    ["In-sample data (75%)", rng_str(is_months), len(is_months), "all choices (incl. the Model 3 threshold) use only this"],
    ["Out-of-sample data (25%)", rng_str(oos_months), len(oos_months), "held out; same fixed rules"],
    ["First formation window", rng_str(months[:FORM_M]), FORM_M, "no returns yet"],
    ["Trading months scored IS", rng_str(months[FORM_M:n_is]), n_is - FORM_M, "returns counted as in-sample"],
    ["Trading months scored OOS", rng_str(oos_months), len(oos_months), "returns counted as out-of-sample"],
], columns=["Window", "Dates", "Months", "Use"]))

SUBPERIODS = [("2019–2020", "2019-03", "2020-12"), ("2021–2022", "2021-01", "2022-12"),
              ("2023 – Jul 2024", "2023-01", str(is_months[-1])), ("OOS", str(OOS0), str(months[-1]))]

,Window,Dates,Months,Use
0,Full data period,Mar 2018 – Sep 2026,103,formation + trading
1,In-sample data (75%),Mar 2018 – Jul 2024,77,all choices (incl. the Model 3 threshold) use ...
2,Out-of-sample data (25%),Aug 2024 – Sep 2026,26,held out; same fixed rules
3,First formation window,Mar 2018 – Feb 2019,12,no returns yet
4,Trading months scored IS,Mar 2019 – Jul 2024,65,returns counted as in-sample
5,Trading months scored OOS,Aug 2024 – Sep 2026,26,returns counted as out-of-sample


**How the split works.** The Gatev loop runs through the whole sample: every month a new cohort forms pairs on the previous 12 months and trades them for 6 months. Each monthly return is labelled IS or OOS by its calendar month, the way Gatev scored his own 1999–2002 holdout (section 1.2). A cohort formed just before the split uses only past data, so nothing leaks; what must stay out of OOS are *choices* (the threshold), and those use IS months only.

## 4. The engine (shared by all models)
Gatev rules: liquidity screen (no day without a price or volume), price index normalised to 1 at the start of formation and continued into trading, best-partner SSD matching, open at the entry threshold × formation SD, close at the next crossing or the last day, delisting (first no-trade day) closes the pair, \$1 long / \$1 short, legs marked to market daily, payoffs reinvested, committed-capital and fully invested returns, overlapping cohorts averaged per month. A pair whose capital reaches zero (short squeeze) is closed – a crypto addition Gatev never needed.

In [4]:
# ---------- formation ----------
def screen(f0, f1):
    return np.where(TRADEABLE[f0:f1].all(axis=0))[0]

def ssd_matrix(f0, f1, cols):
    X = P[f0:f1, cols] / P[f0, cols]
    sq = (X ** 2).sum(0)
    D = np.maximum(sq[:, None] + sq[None, :] - 2 * X.T @ X, 0); np.fill_diagonal(D, np.inf)
    return X, D

def best_partner_pairs(X, D, cols):
    best = {}
    for i in range(len(cols)):
        j = int(D[i].argmin())
        if np.isfinite(D[i, j]):
            a, b = sorted((i, j)); best[(a, b)] = D[a, b]
    return [dict(a=cols[a], b=cols[b], ssd=float(s), sd=float((X[:, a] - X[:, b]).std(ddof=1)))
            for (a, b), s in sorted(best.items(), key=lambda kv: (kv[1], kv[0]))]

def cluster_window(f0, f1, cols, n_components=N_COMPONENTS):
    """qf627 recipe B: z-scored daily returns -> PCA -> standardised loadings -> affinity propagation (defaults)."""
    R = P[f0:f1, cols]; r = R[1:] / R[:-1] - 1
    z = (r - r.mean(0)) / r.std(0, ddof=1)
    pca = PCA(n_components=n_components, random_state=SEED).fit(z)
    XL = StandardScaler().fit_transform(pca.components_.T)
    damping = 0.5
    ap = AffinityPropagation(random_state=SEED).fit(XL)
    while len(ap.cluster_centers_indices_) == 0 and damping < 0.95:      # only if AP fails to converge
        damping += 0.1
        ap = AffinityPropagation(random_state=SEED, damping=damping, max_iter=1000).fit(XL)
    return dict(labels=ap.labels_, exemplars=ap.cluster_centers_indices_, pca=pca, loadings=pca.components_.T,
                XL=XL, z=z, damping=damping, n_iter=ap.n_iter_)

def within(X, D, cols, labels):
    D = D.copy(); D[labels[:, None] != labels[None, :]] = np.inf
    return best_partner_pairs(X, D, cols)

# ---------- trading ----------
def pair_trades(p, f0, t0, t1, wait, entry_sd):
    """Executed trades (open_day, close_day, long_col, short_col), days relative to t0."""
    a, b = p["a"], p["b"]
    ok = TRADEABLE[t0:t1, a] & TRADEABLE[t0:t1, b]
    T = t1 - t0
    L = T - 1 if ok.all() else int(np.argmin(ok)) - 1          # last usable day (delisting = first no-trade day)
    if L < 1:
        return []
    s = P[t0:t0 + L + 1, a] / P[f0, a] - P[t0:t0 + L + 1, b] / P[f0, b]   # index continues from formation start
    thr = entry_sd * p["sd"]
    sig = np.zeros(L + 1, int); state = 0
    for t in range(L + 1):
        v = s[t]
        if state == 0:   state = -1 if v > thr else (1 if v < -thr else 0)
        elif state == -1 and v <= 0: state = 0                  # crossing
        elif state == 1 and v >= 0:  state = 0
        sig[t] = state
    pos = np.zeros(L + 1, int)
    pos[wait:] = sig[:L + 1 - wait]                              # execute `wait` days after the signal
    pos[L] = 0                                                   # forced close on the last usable day
    trades, t = [], 0
    while t <= L:
        if pos[t] != 0:
            o, d = t, pos[t]
            while pos[t] == d: t += 1
            trades.append((o, t, *((b, a) if d == -1 else (a, b))))   # d=-1: A rich -> long B, short A
        else:
            t += 1
    return trades

TCACHE = {}
def trades_for(c, p, sd, wait):
    key = (c["k"], p["a"], p["b"], sd, wait)
    if key not in TCACHE:
        TCACHE[key] = pair_trades(p, c["f0"], c["t0"], c["t1"], wait, sd)
    return TCACHE[key]

def pair_path(trades, t0, T, cost):
    """Daily value of one pair's $1 of committed capital with reinvested payoffs (Gatev eq. 2-3)."""
    Vend = np.empty(T); cur = 1.0; t = 0; bust = None
    for (o, c, lg, sh) in trades:
        Vend[t:o] = cur; V0 = cur
        rl = P[t0 + o:t0 + c + 1, lg] / P[t0 + o, lg]
        rs = P[t0 + o:t0 + c + 1, sh] / P[t0 + o, sh]
        path = V0 * (1 - 2 * cost + (rl - 1) - (rs - 1))
        path[-1] -= cost * V0 * (rl[-1] + rs[-1])
        neg = np.where(path[1:] <= 0)[0]
        if len(neg):                                             # short squeeze wiped out the pair's capital
            k = neg[0] + 1; path = path[:k + 1]; c = o + k; bust = c
        Vend[o:c + 1] = path; cur = path[-1]; t = c + 1
        if bust is not None: break
    Vend[t:] = cur if bust is None else 0.0
    Vprev = np.concatenate([[1.0], Vend[:-1]])
    dV = Vend - Vprev
    if bust is not None and bust + 1 < T:
        dV[bust + 1:] = 0.0; Vprev[bust + 1:] = 0.0
    return dV, Vprev, bust is not None

def port_returns(dVs, Vps, traded, day_months):
    dV, Vp = np.array(dVs), np.array(Vps)
    with np.errstate(invalid="ignore", divide="ignore"):
        rc = np.where(Vp.sum(0) > 0, dV.sum(0) / Vp.sum(0), 0.0)
        tr = np.array(traded)
        rf = np.where(Vp[tr].sum(0) > 0, dV[tr].sum(0) / Vp[tr].sum(0), 0.0) if tr.any() else np.zeros(dV.shape[1])
    to_m = lambda r: pd.Series(1 + r).groupby(day_months).prod() - 1
    return to_m(rc), to_m(rf)

def run(cohorts, sd=BASE_SD, wait=1, cost_bp=0.0, coin_map=None, log=True):
    """Committed capital: all selected pairs; fully invested: pairs that opened. If a cohort has fewer pairs than a
    portfolio needs, the portfolio holds all available pairs."""
    cost = cost_bp / 1e4
    comm = {n: {} for n in PORTS}; full = {n: {} for n in PORTS}
    plog, busts, short = [], 0, {n: 0 for n in PORTS}
    for c in cohorts:
        T = c["t1"] - c["t0"]; dVs, Vps, traded = [], [], []
        for r, p in enumerate(c["pairs"]):
            tr = trades_for(c, p, sd, wait) if coin_map is None else coin_map[(c["k"], r)]
            dV, Vp, b = pair_path(tr, c["t0"], T, cost)
            busts += b; dVs.append(dV); Vps.append(Vp); traded.append(len(tr) > 0)
            if log:
                plog.append(dict(cohort=str(c["start"]), complete=c["complete"], rank=r + 1, coin_a=universe[p["a"]],
                                 coin_b=universe[p["b"]], ssd=p["ssd"], trigger=sd * p["sd"], round_trips=len(tr),
                                 days_open=sum(t[1] - t[0] for t in tr), payoff=float(dV.sum())))
        if not c["pairs"]: continue
        for n, N in PORTS.items():
            N = len(c["pairs"]) if N is None else N
            short[n] += len(c["pairs"]) < N
            N = min(N, len(c["pairs"]))
            comm[n][c["k"]], full[n][c["k"]] = port_returns(dVs[:N], Vps[:N], traded[:N], c["day_months"])
    stag = lambda d: pd.DataFrame({n: pd.concat(v, axis=1).mean(axis=1) for n, v in d.items() if v})
    return dict(comm=stag(comm), full=stag(full), log=pd.DataFrame(plog), busts=busts, short=short,
                ncoh=pd.concat(comm["Top 5"], axis=1).notna().sum(axis=1))

# ---------- statistics ----------
def nw_se(x, lags=NW_LAGS):
    x = np.asarray(pd.Series(x).dropna(), float); T = len(x); u = x - x.mean(); v = u @ u / T
    for l in range(1, min(lags, T - 1) + 1):
        v += 2 * (1 - l / (lags + 1)) * (u[l:] @ u[:-l]) / T
    return math.sqrt(v / T)
def tstat(x): x = pd.Series(x).dropna(); se = nw_se(x); return x.mean() / se if se > 0 else np.nan
def sharpe(x): x = pd.Series(x).dropna(); return x.mean() / x.std() * np.sqrt(12) if x.std() > 0 else np.nan
def mdd(x):
    w = np.concatenate([[1.0], (1 + pd.Series(x).dropna()).cumprod().values]); return (w / np.maximum.accumulate(w) - 1).min()
def summ(x):
    x = pd.Series(x).dropna()
    return {"Months": len(x), "Mean / month": x.mean(), "t (NW)": tstat(x), "Sharpe": sharpe(x), "Max drawdown": mdd(x),
            "Months < 0": (x < 0).mean()}
def fmt(df, pct=("Mean / month", "Max drawdown", "Months < 0", "Mean / month (committed)", "Mean / month (fully invested)",
                 "Difference / month", "Alpha", "Mean")):
    out = df.copy().astype(object)
    for c in df.columns:
        if c in pct: out[c] = [f"{v:.2%}" if pd.notna(v) else "" for v in df[c]]
        elif df[c].dtype.kind == "f": out[c] = [f"{v:.2f}" if pd.notna(v) else "" for v in df[c]]
    return out

### Build the cohorts for Models 1 and 2

In [5]:
COH = {"Model 1": [], "Model 2": []}
for k in range(n_m - FORM_M):
    f0, f1 = mstart[k], mstart[k + FORM_M]
    t1 = mstart[min(k + FORM_M + TRADE_M, n_m)]
    cols = screen(f0, f1)
    X, D = ssd_matrix(f0, f1, cols)
    cl = cluster_window(f0, f1, cols)
    base = dict(k=k, start=months[k + FORM_M], f0=f0, t0=f1, t1=t1, cols=cols, n_coins=len(cols), X=X, D=D,
                complete=k + FORM_M + TRADE_M <= n_m, day_months=dates[f1:t1].to_period("M"))
    COH["Model 1"].append({**base, "pairs": best_partner_pairs(X, D, cols), "cl": None})
    COH["Model 2"].append({**base, "pairs": within(X, D, cols, cl["labels"]), "cl": cl})
print(f"{len(COH['Model 1'])} cohorts. Coins passing the screen per window: min {min(c['n_coins'] for c in COH['Model 1'])}, "
      f"median {int(np.median([c['n_coins'] for c in COH['Model 1']]))}, max {max(c['n_coins'] for c in COH['Model 1'])}")
for m in COH:
    n = [len(c["pairs"]) for c in COH[m]]
    print(f"{m}: pairs per cohort min {min(n)}, median {int(np.median(n))}, max {max(n)}")

91 cohorts. Coins passing the screen per window: min 24, median 35, max 39
Model 1: pairs per cohort min 19, median 28, max 34
Model 2: pairs per cohort min 17, median 23, max 30


### Hand-checked test of the engine

In [6]:
_P, _TR = P, TRADEABLE
P = np.array([[100, 100], [100, 100], [100, 100], [100, 80], [100, 80], [100, 100], [100, 100.0]])
TRADEABLE = np.ones_like(P, bool)
p = dict(a=0, b=1, sd=0.05)                        # threshold 0.10 at 2 SD; B drops 20% -> divergence
for w in (0, 1):
    tr = pair_trades(p, 0, 1, 7, w, 2.0); dV, _, _ = pair_path(tr, 1, 6, 0.0)
    assert len(tr) == 1 and tr[0][2] == 1 and abs(dV.sum() - 0.25) < 1e-12
assert pair_trades(p, 0, 1, 7, 0, 5.0) == []       # 5 SD = 0.25 > 0.20 divergence -> no trade
dV, _, _ = pair_path(pair_trades(p, 0, 1, 7, 0, 2.0), 1, 6, 0.001)
assert abs(dV.sum() - (0.25 - 0.002 - 0.001 * 2.25)) < 1e-12
P, TRADEABLE = _P, _TR
print("Engine tests passed: +25% on $1 when the laggard recovers 80 -> 100; no trade above the threshold; fees per leg.")

Engine tests passed: +25% on $1 when the laggard recovers 80 -> 100; no trade above the threshold; fees per leg.


# Part A – Model 1: Gatev baseline

In [7]:
RES = {}
for m in ["Model 1", "Model 2"]:
    for w in (0, 1):
        RES[(m, w)] = run(COH[m], BASE_SD, w)
r = RES[("Model 1", 1)]
print(f"Monthly returns: {r['comm'].index[0]} to {r['comm'].index[-1]} ({len(r['comm'])} months); "
      f"overlapping cohorts per month: min {r['ncoh'].min()}, max {r['ncoh'].max()} (fewer only in the first 5 months)")
for m in ["Model 1", "Model 2"]:
    R = RES[(m, 1)]
    print(f"{m}: pairs wiped out by a short squeeze {R['busts']} of {len(R['log'])} pair-cohorts; "
          f"cohorts with fewer pairs than the portfolio size {R['short']}")

Monthly returns: 2019-03 to 2026-09 (91 months); overlapping cohorts per month: min 1, max 6 (fewer only in the first 5 months)
Model 1: pairs wiped out by a short squeeze 312 of 2507 pair-cohorts; cohorts with fewer pairs than the portfolio size {'Top 5': 0, 'Top 20': 1, 'All': 0}
Model 2: pairs wiped out by a short squeeze 249 of 2104 pair-cohorts; cohorts with fewer pairs than the portfolio size {'Top 5': 0, 'Top 20': 8, 'All': 0}


### A1. Table 1 replica (full sample, 2 SD)

In [8]:
def table1(R):
    out = {}
    for n in PORTS:
        f, c = R["full"][n].dropna(), R["comm"][n].dropna(); z = (f - f.mean()) / f.std(ddof=0)
        out[n] = {"Average excess return (fully invested)": f.mean(), "Standard error (Newey-West)": nw_se(f),
                  "t-statistic": tstat(f), "Median": f.median(), "Standard deviation": f.std(), "Skewness": (z ** 3).mean(),
                  "Kurtosis": (z ** 4).mean(), "Minimum": f.min(), "Maximum": f.max(),
                  "Observations with excess return < 0": (f < 0).mean(),
                  "Average excess return on committed capital": c.mean(), "t-statistic (committed)": tstat(c)}
    t = pd.DataFrame(out); o = t.copy().astype(object)
    for row in t.index:
        f_ = "{:.0%}" if "< 0" in row else ("{:.2f}" if row.startswith(("t-stat", "Skew", "Kurt")) else "{:.4f}")
        o.loc[row] = [f_.format(v) for v in t.loc[row]]
    return o

def period_table(R):
    rows = []
    for n in PORTS:
        for lab, sel in [("In-sample", is_m), ("Out-of-sample", oos_m), ("Full", lambda i: i == i)]:
            f, c = R["full"][n], R["comm"][n]; f, c = f[sel(f.index)], c[sel(c.index)]
            rows.append({"Portfolio": n, "Period": lab, "Dates": rng_str(c.index), "Months": len(c),
                         "Mean / month (committed)": c.mean(), "t (NW) committed": tstat(c), "Sharpe (committed)": sharpe(c),
                         "Mean / month (fully invested)": f.mean(), "t (NW) fully": tstat(f), "Sharpe (fully)": sharpe(f),
                         "Max drawdown": mdd(c), "Months < 0": (c < 0).mean()})
    return fmt(pd.DataFrame(rows).set_index(["Portfolio", "Period"]))

print("Model 1, Panel A – no waiting"); display(table1(RES[("Model 1", 0)]))
print("Model 1, Panel B – one-day waiting (headline)"); display(table1(RES[("Model 1", 1)]))

Model 1, Panel A – no waiting


,Top 5,Top 20,All
Average excess return (fully invested),0.0226,0.0216,0.0150
Standard error (Newey-West),0.0086,0.0099,0.0071
t-statistic,2.64,2.19,2.10
Median,0.0310,0.0200,0.0167
Standard deviation,0.1014,0.0779,0.0615
Skewness,0.55,1.62,0.57
Kurtosis,6.42,9.71,7.09
Minimum,-0.2643,-0.1712,-0.1881
Maximum,0.4375,0.4022,0.2834
Observations with excess return < 0,35%,36%,35%


Model 1, Panel B – one-day waiting (headline)


,Top 5,Top 20,All
Average excess return (fully invested),0.0173,0.0148,0.0090
Standard error (Newey-West),0.0067,0.0082,0.0063
t-statistic,2.60,1.81,1.43
Median,0.0142,0.0162,0.0134
Standard deviation,0.1061,0.0743,0.0591
Skewness,1.96,1.14,0.35
Kurtosis,14.60,6.45,5.00
Minimum,-0.2991,-0.1416,-0.1694
Maximum,0.6320,0.3119,0.2374
Observations with excess return < 0,41%,38%,42%


### A2. Model 1 in-sample vs out-of-sample (one-day wait)

In [9]:
display(period_table(RES[("Model 1", 1)]))

Dates Months Mean / month (committed) t (NW) committed Sharpe (committed) Mean / month (fully invested) t (NW) fully Sharpe (fully) Max drawdown Months < 0
Portfolio Period                                                                                                                                                                                  
Top 5     In-sample      Mar 2019 – Jul 2024     65                    1.46%             1.90               0.48                         2.01%         2.40           0.61      -39.62%     38.46%
          Out-of-sample  Aug 2024 – Sep 2026     26                    1.25%             1.59               0.63                         1.04%         1.10           0.45      -14.52%     50.00%
          Full           Mar 2019 – Sep 2026     91                    1.40%             2.37               0.51                         1.73%         2.60           0.57      -39.62%     41.76%
Top 20    In-sample      Mar 2019 – Jul 2024     65                    1.59%             1.59               0.79                         1.97%         1.83           0.86      -33.51%     40.00%
          Out-of-sample  Aug 2024 – Sep 2026     26                    0.08%             0.19               0.06                         0.26%         0.46           0.16      -18.00%     38.46%
          Full           Mar 2019 – Sep 2026     91                    1.16%             1.54               0.63                         1.48%         1.81           0.69      -34.60%     39.56%
All       In-sample      Mar 2019 – Jul 2024     65                    0.96%             1.28               0.69                         1.30%         1.59           0.79      -23.58%     41.54%
          Out-of-sample  Aug 2024 – Sep 2026     26                   -0.13%            -0.35              -0.09                        -0.09%        -0.18          -0.05      -18.58%     46.15%
          Full           Mar 2019 – Sep 2026     91                    0.65%             1.15               0.46                         0.90%         1.43           0.53      -29.03%     42.86%

### A3. Table 2 replica – trading statistics (complete cohorts, one-day wait)

In [10]:
def table2(R):
    L = R["log"]; L = L[L.complete]; out = {}
    for n, N in PORTS.items():
        g = L if N is None else L[L["rank"] <= N]; traded = g[g.round_trips > 0]
        out[n] = {"Average price deviation trigger for opening pairs": g.trigger.mean(),
                  "Average number of pairs traded per six-month period": g.groupby("cohort").round_trips.apply(lambda s: (s > 0).sum()).mean(),
                  "Average number of round-trip trades per pair": traded.round_trips.mean(),
                  "Average time pairs are open in months": (traded.days_open / (365 / 12)).mean(),
                  "Pairs selected per period": g.groupby("cohort").size().mean()}
    return pd.DataFrame(out).round(3)
display(pd.concat({m: table2(RES[(m, 1)]) for m in ["Model 1", "Model 2"]}, axis=1))

Model 1                 Model 2                
                                                     Top 5  Top 20     All   Top 5  Top 20     All
Average price deviation trigger for opening pairs    0.218   0.351   0.585   0.245   0.495   0.719
Average number of pairs traded per six-month pe...   3.953  15.686  21.093   3.907  15.244  17.547
Average number of round-trip trades per pair         1.794   1.664   1.594   1.824   1.555   1.516
Average time pairs are open in months                3.831   3.943   3.944   4.220   4.116   4.185
Pairs selected per period                            5.000  20.000  27.709   5.000  19.872  23.244

# Part B – Model 2: PCA + affinity propagation, then Gatev pairs within clusters
Inside every formation window: z-scored daily returns → PCA with 5 components → standardised loadings → `AffinityPropagation(random_state=SEED)` with scikit-learn defaults (qf627 Week 7, recipe B). Each coin's partner must come from its own cluster; everything else is Model 1.

### B1. Clustering diagnostics across all windows

In [11]:
diag = []
for c, c1 in zip(COH["Model 2"], COH["Model 1"]):
    sizes = pd.Series(c["cl"]["labels"]).value_counts()
    diag.append({"cohort": c["start"], "coins": c["n_coins"], "clusters": len(sizes), "singletons": int((sizes == 1).sum()),
                 "largest cluster": int(sizes.max()), "within-cluster pairs": len(c["pairs"]), "Model 1 pairs": len(c1["pairs"]),
                 "variance explained (5 PCs)": c["cl"]["pca"].explained_variance_ratio_.sum(),
                 "AP damping used": c["cl"]["damping"]})
diag = pd.DataFrame(diag)
ari = [np.nan]
for a, b in zip(COH["Model 2"][:-1], COH["Model 2"][1:]):
    common = np.intersect1d(a["cols"], b["cols"])
    ari.append(adjusted_rand_score(pd.Series(a["cl"]["labels"], index=a["cols"])[common],
                                   pd.Series(b["cl"]["labels"], index=b["cols"])[common]))
diag["ARI vs previous window"] = ari
display(diag.drop(columns="cohort").describe().loc[["mean", "min", "50%", "max"]].round(2))
print(f"Windows with fewer than 20 within-cluster pairs: {(diag['within-cluster pairs'] < 20).sum()} of {len(diag)}; "
      f"AP converged with default damping in {(diag['AP damping used'] == 0.5).sum()} of {len(diag)}")

over = (diag.assign(date=diag.cohort.dt.to_timestamp())[["date", "clusters", "singletons", "within-cluster pairs", "Model 1 pairs"]]
        .melt("date", var_name="series", value_name="count"))
p = (ggplot() + geom_line(aes("date", "count", color="series"), data=over, size=1.1)
     + geom_vline(xintercept=OOS0.to_timestamp(), linetype="dashed", color=INK2)
     + scale_color_manual(values=[BLUE, ORANGE, AQUA, MUTED], breaks=["clusters", "singletons", "within-cluster pairs", "Model 1 pairs"])
     + scale_x_datetime(format="%Y")
     + labs(title="AP clusters, singleton coins and pairs per formation window", subtitle="dashed line = start of out-of-sample",
            x="cohort start (first trading month)", y="count", color="")
     + theme_minimal() + theme(legend_position="top") + ggsize(900, 380))
show(p, "B1_clusters_over_time")

st = diag.assign(date=diag.cohort.dt.to_timestamp()).dropna(subset=["ARI vs previous window"])
p = (ggplot() + geom_line(aes("date", "ARI vs previous window"), data=st, color=BLUE, size=1.1)
     + geom_hline(yintercept=st["ARI vs previous window"].mean(), linetype="dashed", color=INK2)
     + scale_x_datetime(format="%Y")
     + labs(title="Cluster stability: adjusted Rand index (ARI) between consecutive windows",
            subtitle=f"1 = identical clusters, 0 = no more alike than chance; dashed = mean {st['ARI vs previous window'].mean():.2f}",
            x="cohort start", y="ARI")
     + theme_minimal() + ggsize(900, 320))
show(p, "B1_cluster_stability")

,coins,clusters,singletons,largest cluster,within-cluster pairs,Model 1 pairs,variance explained (5 PCs),AP damping used,ARI vs previous window
mean,34.31,7.98,2.86,12.54,23.12,27.55,0.64,0.5,0.58
min,24.00,5.00,0.00,7.00,17.00,19.00,0.55,0.5,0.12
50%,35.00,8.00,3.00,12.00,23.00,28.00,0.64,0.5,0.59
max,39.00,11.00,8.00,21.00,30.00,34.00,0.71,0.5,1.00


Windows with fewer than 20 within-cluster pairs: 8 of 91; AP converged with default damping in 91 of 91


Fontconfig error: Cannot load default config file: No such file: (null)


### B2. One formation window in detail: PCA
The **last in-sample formation window** is shown. Change `SHOW_K` to inspect another window.

In [12]:
SHOW_K = n_is - FORM_M - 1
cw = COH["Model 2"][SHOW_K]; cl = cw["cl"]; names = [universe[i] for i in cw["cols"]]
f_lo, f_hi = months[cw["k"]], months[cw["k"] + FORM_M - 1]
print(f"Window shown: formation {f_lo.strftime('%b %Y')} – {f_hi.strftime('%b %Y')} ({len(names)} coins); pairs trade from {cw['start'].strftime('%b %Y')}")

pca_full = PCA(random_state=SEED).fit(cl["z"])
display(pd.DataFrame({"share": cl["pca"].explained_variance_ratio_, "cumulative": cl["pca"].explained_variance_ratio_.cumsum()},
                     index=[f"PC{k}" for k in range(1, N_COMPONENTS + 1)]).round(4))
ew = (P[cw["f0"] + 1:cw["t0"], cw["cols"]] / P[cw["f0"]:cw["t0"] - 1, cw["cols"]] - 1).mean(axis=1)
load = pd.DataFrame(cl["loadings"], index=names, columns=[f"PC{k}" for k in range(1, N_COMPONENTS + 1)])
print("components needed for 90% of the variance      :", int(np.searchsorted(pca_full.explained_variance_ratio_.cumsum(), 0.9) + 1), "of", len(names))
print("corr of PC1's daily return with equal-weight mkt:", round(abs(np.corrcoef(cl["pca"].transform(cl["z"])[:, 0], ew)[0, 1]), 4))
for k in [2, 3]:
    rk = load[f"PC{k}"].sort_values()
    print(f"PC{k}, three lowest / highest loadings        :", rk.index[:3].tolist(), "/", rk.index[-3:].tolist())

n_show = min(20, len(names))
scree = pd.DataFrame({"component": np.arange(1, n_show + 1), "share": pca_full.explained_variance_ratio_[:n_show]})
scree["cumulative"] = scree["share"].cumsum()
p = (ggplot(scree, aes(x="component"))
     + geom_bar(aes(y="share"), stat="identity", fill=BLUE, width=0.75)
     + geom_line(aes(y="cumulative"), color=ORANGE, size=1) + geom_point(aes(y="cumulative"), color=ORANGE, size=2.5)
     + geom_vline(xintercept=N_COMPONENTS + 0.5, linetype="dashed", color=RED)
     + scale_y_continuous(format=".0%") + scale_x_continuous(breaks=list(range(1, n_show + 1)))
     + labs(title=f"PCA scree: {len(names)} coins' standardised daily returns, {f_lo.strftime('%b %Y')} – {f_hi.strftime('%b %Y')}",
            subtitle=f"bars = share of variance, line = running total, dashed = the {N_COMPONENTS} components kept",
            x="principal component", y="share of total variance")
     + theme_minimal() + ggsize(900, 420))
show(p, "B2_pca_scree")

lp = load.reset_index().rename(columns={"index": "coin"}).melt("coin", var_name="component", value_name="loading")
p = (ggplot(lp, aes("component", "coin", fill="loading")) + geom_tile(color="white")
     + scale_fill_gradient2(low=BLUE, mid="#f0efec", high=RED, midpoint=0)
     + labs(title="PCA loadings per coin", subtitle="the AP features, before standardising", x="", y="")
     + theme_minimal() + ggsize(560, 760))
show(p, "B2_pca_loadings")

Window shown: formation Jul 2023 – Jun 2024 (32 coins); pairs trade from Jul 2024


,share,cumulative
PC1,0.4066,0.4066
PC2,0.0515,0.4581
PC3,0.0377,0.4959
PC4,0.0361,0.5320
PC5,0.0347,0.5667


components needed for 90% of the variance      : 20 of 32
corr of PC1's daily return with equal-weight mkt: 0.9847
PC2, three lowest / highest loadings        : ['STEEM', 'REP', 'XVG'] / ['ADA', 'XLM', 'XRP']
PC3, three lowest / highest loadings        : ['NMC', 'SC', 'BTC'] / ['BCH', 'DGB', 'XVG']


### B3. Affinity propagation in the same window
Each coin is drawn at its first two standardised loadings and joined to its **exemplar** (ringed), the coin AP elected to represent its cluster. AP uses all five loadings, so two coins can look close here and still sit in different clusters.

In [13]:
lab = cl["labels"]; ex = cl["exemplars"]
pts = pd.DataFrame({"coin": names, "PC1 loading": cl["XL"][:, 0], "PC2 loading": cl["XL"][:, 1], "cluster": [f"C{l + 1}" for l in lab]})
pts["exemplar_x"] = cl["XL"][ex[lab], 0]; pts["exemplar_y"] = cl["XL"][ex[lab], 1]; pts["exemplar"] = [names[e] for e in ex[lab]]
exp_ = pts.iloc[ex]
order_cl = pts.cluster.value_counts().index.tolist()
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
p = (ggplot()
     + geom_segment(aes(x="PC1 loading", y="PC2 loading", xend="exemplar_x", yend="exemplar_y", color="cluster"), data=pts, size=0.6, alpha=0.7)
     + geom_point(aes(x="PC1 loading", y="PC2 loading", color="cluster"), data=pts, size=3.2)
     + geom_point(aes(x="PC1 loading", y="PC2 loading"), data=exp_, color=INK, size=6.5, shape=21, fill="white", stroke=1.6)
     + geom_point(aes(x="PC1 loading", y="PC2 loading", color="cluster"), data=exp_, size=3.2)
     + geom_text_repel(aes(x="PC1 loading", y="PC2 loading", label="coin"), data=pts, size=4, color=INK2)
     + scale_color_manual(values=[PAL[i] if i < len(PAL) else MUTED for i in range(len(order_cl))], breaks=order_cl)
     + coord_fixed()
     + labs(title=f"Affinity propagation on PCA loadings, {f_lo.strftime('%b %Y')} – {f_hi.strftime('%b %Y')}: {len(ex)} clusters",
            subtitle="lines join each coin to its exemplar (ringed); AP uses all 5 loadings",
            x="PC1 loading, standardised", y="PC2 loading, standardised", color="cluster")
     + theme_minimal() + ggsize(900, 760))
show(p, "B3_ap_scatter")

members = (pts.groupby("cluster").agg(exemplar=("exemplar", "first"), size=("coin", "size"), coins=("coin", lambda s: ", ".join(sorted(s))))
           .sort_values("size", ascending=False))
print("Cluster membership in this window:"); display(members)

m1_top = {frozenset((universe[q["a"]], universe[q["b"]])) for q in COH["Model 1"][SHOW_K]["pairs"][:20]}
wp = pd.DataFrame([dict(rank=i + 1, coin_a=universe[q["a"]], coin_b=universe[q["b"]], ssd=round(q["ssd"], 3),
                        cluster=pts.set_index("coin").loc[universe[q["a"]], "cluster"],
                        **{"also in Model 1 Top 20": frozenset((universe[q["a"]], universe[q["b"]])) in m1_top})
                   for i, q in enumerate(cw["pairs"])])
print(f"Within-cluster pairs formed in this window ({len(wp)}), ranked by SSD:"); display(wp.head(20).set_index("rank"))

big = members.index[0]; mem = members.loc[big, "coins"].split(", "); idx = [universe.index(x) for x in mem]
fp = pd.DataFrame(P[cw["f0"]:cw["t0"], idx] / P[cw["f0"], idx], index=dates[cw["f0"]:cw["t0"]], columns=mem).iloc[::3]
fp = fp.rename_axis("date").reset_index().melt("date", var_name="coin", value_name="normalised price")
p = (ggplot() + geom_hline(yintercept=1, color=MUTED)
     + geom_line(aes("date", "normalised price", group="coin"), data=fp, color=BLUE, alpha=0.55, size=0.8)
     + geom_text_repel(aes("date", "normalised price", label="coin"), data=fp.groupby("coin").tail(1), size=3.5, color=INK2)
     + scale_y_log10() + scale_x_datetime(format="%b %Y")
     + labs(title=f"Formation-window prices of the largest cluster ({big}, {len(mem)} coins), normalised to 1, log scale",
            subtitle=", ".join(mem), x="", y="normalised price")
     + theme_minimal() + ggsize(900, 380))
show(p, "B3_cluster_paths")

IOStream.flush timed out


Cluster membership in this window:


,exemplar,size,coins
cluster,,,
C3,ETH,7,"BNB, BTC, DOGE, ETC, ETH, NEO, QTUM"
C4,LINK,7,"ADA, DASH, LINK, TRX, WAVES, XMR, ZEC"
C2,BCH,6,"BCH, DCR, DGB, KMD, LTC, SYS"
C6,VTC,5,"BTS, PIVX, REP, SC, VTC"
C1,ARK,3,"ARK, LSK, STEEM"
C7,XRP,2,"XLM, XRP"
C5,NMC,1,NMC
C8,XVG,1,XVG


Within-cluster pairs formed in this window (21), ranked by SSD:


,coin_a,coin_b,ssd,cluster,also in Model 1 Top 20
rank,,,,,
1,DASH,ZEC,1.985,C4,True
2,XLM,XRP,4.983,C7,True
3,ETC,QTUM,6.875,C3,False
4,XMR,ZEC,10.144,C4,True
5,DGB,SYS,12.692,C2,True
6,BCH,KMD,13.976,C2,True
7,ETC,NEO,16.161,C3,False
8,BTC,DOGE,16.811,C3,True
9,DCR,KMD,16.914,C2,False


### B4. Model 2 results (2 SD) and comparison with Model 1

In [14]:
print("Model 2, Panel A – no waiting"); display(table1(RES[("Model 2", 0)]))
print("Model 2, Panel B – one-day waiting (headline)"); display(table1(RES[("Model 2", 1)]))
display(period_table(RES[("Model 2", 1)]))

rows = []
for n in PORTS:
    for lab, sel in [("In-sample", is_m), ("Out-of-sample", oos_m)]:
        a, b = RES[("Model 1", 1)]["comm"][n], RES[("Model 2", 1)]["comm"][n]
        d = (b - a).dropna(); d = d[sel(d.index)]
        rows.append({"Portfolio": n, "Period": lab, "Model 1 mean": a[sel(a.index)].mean(), "Model 2 mean": b[sel(b.index)].mean(),
                     "Difference / month": d.mean(), "t (NW) of difference": tstat(d),
                     "Model 1 Sharpe": sharpe(a[sel(a.index)]), "Model 2 Sharpe": sharpe(b[sel(b.index)])})
cmp12 = pd.DataFrame(rows).set_index(["Portfolio", "Period"])
display(fmt(cmp12, pct=("Model 1 mean", "Model 2 mean", "Difference / month")))
ov = [len({frozenset((q["a"], q["b"])) for q in a["pairs"][:20]} & {frozenset((q["a"], q["b"])) for q in b["pairs"][:20]}) / min(20, len(b["pairs"]))
      for a, b in zip(COH["Model 1"], COH["Model 2"])]
print(f"Share of Model 2's Top 20 pairs also in Model 1's Top 20: mean {np.mean(ov):.0%} (min {min(ov):.0%}, max {max(ov):.0%})")

Model 2, Panel A – no waiting


,Top 5,Top 20,All
Average excess return (fully invested),0.0133,0.0160,0.0131
Standard error (Newey-West),0.0120,0.0076,0.0074
t-statistic,1.11,2.09,1.77
Median,0.0201,0.0168,0.0165
Standard deviation,0.1405,0.0727,0.0678
Skewness,0.07,2.21,2.15
Kurtosis,12.70,14.66,13.33
Minimum,-0.6282,-0.1537,-0.1238
Maximum,0.7090,0.4397,0.3969
Observations with excess return < 0,40%,35%,42%


Model 2, Panel B – one-day waiting (headline)


,Top 5,Top 20,All
Average excess return (fully invested),0.0130,0.0139,0.0105
Standard error (Newey-West),0.0121,0.0071,0.0069
t-statistic,1.07,1.96,1.52
Median,0.0203,0.0112,0.0144
Standard deviation,0.1414,0.0747,0.0693
Skewness,-1.80,2.22,2.39
Kurtosis,18.34,12.73,13.39
Minimum,-0.8386,-0.1560,-0.1276
Maximum,0.5694,0.3762,0.3545
Observations with excess return < 0,36%,40%,40%


Dates Months Mean / month (committed) t (NW) committed Sharpe (committed) Mean / month (fully invested) t (NW) fully Sharpe (fully) Max drawdown Months < 0
Portfolio Period                                                                                                                                                                                  
Top 5     In-sample      Mar 2019 – Jul 2024     65                    0.58%             0.39               0.13                         1.25%         0.78           0.28      -89.30%     36.92%
          Out-of-sample  Aug 2024 – Sep 2026     26                    1.38%             1.49               0.59                         1.42%         1.06           0.47      -15.77%     38.46%
          Full           Mar 2019 – Sep 2026     91                    0.81%             0.74               0.21                         1.30%         1.07           0.32      -89.30%     37.36%
Top 20    In-sample      Mar 2019 – Jul 2024     65                    1.40%             1.64               0.64                         1.74%         1.85           0.73      -24.42%     38.46%
          Out-of-sample  Aug 2024 – Sep 2026     26                    0.35%             0.84               0.35                         0.52%         0.89           0.37      -13.49%     42.31%
          Full           Mar 2019 – Sep 2026     91                    1.10%             1.72               0.57                         1.39%         1.96           0.64      -24.42%     39.56%
All       In-sample      Mar 2019 – Jul 2024     65                    1.19%             1.38               0.58                         1.47%         1.64           0.67      -18.39%     38.46%
          Out-of-sample  Aug 2024 – Sep 2026     26                    0.08%             0.18               0.07                        -0.00%        -0.00          -0.00      -15.36%     46.15%
          Full           Mar 2019 – Sep 2026     91                    0.87%             1.35               0.48                         1.05%         1.52           0.52      -18.39%     40.66%

Model 1 mean Model 2 mean Difference / month t (NW) of difference Model 1 Sharpe Model 2 Sharpe
Portfolio Period                                                                                                       
Top 5     In-sample            1.46%        0.58%             -0.89%                -0.56           0.48           0.13
          Out-of-sample        1.25%        1.38%              0.13%                 0.21           0.63           0.59
Top 20    In-sample            1.59%        1.40%             -0.19%                -0.42           0.79           0.64
          Out-of-sample        0.08%        0.35%              0.27%                 0.52           0.06           0.35
All       In-sample            0.96%        1.19%              0.23%                 0.57           0.69           0.58
          Out-of-sample       -0.13%        0.08%              0.21%                 0.43          -0.09           0.07

Share of Model 2's Top 20 pairs also in Model 1's Top 20: mean 39% (min 15%, max 70%)


### B5. Placebo clusters (sharp test of H1)
Does AP find *useful* groups, or does any restriction to fewer candidate pairs do as well? In every window, the AP cluster labels are randomly shuffled across coins (same number and sizes of clusters), pairs are formed within these random clusters with the same Gatev rules, and the strategy is rerun. 100 runs.

In [15]:
rng_pl = np.random.default_rng(SEED + 1)
pl = []
for b in range(N_PLACEBO):
    coh = [{**c, "pairs": within(c["X"], c["D"], c["cols"], rng_pl.permutation(c["cl"]["labels"]))} for c in COH["Model 2"]]
    R = run(coh, BASE_SD, 1, log=False)
    pl.append({(n, lab): R["comm"][n][sel(R["comm"].index)].mean() for n in PORTS for lab, sel in [("IS", is_m), ("OOS", oos_m)]})
PL = pd.DataFrame(pl)
rows = []
for n in PORTS:
    for lab, sel in [("IS", is_m), ("OOS", oos_m)]:
        act = RES[("Model 2", 1)]["comm"][n]; act = act[sel(act.index)].mean()
        rows.append({"Portfolio": n, "Period": lab, "AP clusters (actual)": act, "Random clusters: mean": PL[(n, lab)].mean(),
                     "Random clusters: 95th pct": PL[(n, lab)].quantile(0.95), "Share of random >= actual": (PL[(n, lab)] >= act).mean()})
placebo_tab = pd.DataFrame(rows).set_index(["Portfolio", "Period"])
display(fmt(placebo_tab, pct=("AP clusters (actual)", "Random clusters: mean", "Random clusters: 95th pct")))

d = pd.concat([pd.DataFrame({"mean monthly return": PL[("Top 20", lab)], "period": lab}) for lab in ["IS", "OOS"]])
act = pd.DataFrame({"period": ["IS", "OOS"], "x": [placebo_tab.loc[("Top 20", l), "AP clusters (actual)"] for l in ["IS", "OOS"]]})
p = (ggplot() + geom_histogram(aes("mean monthly return"), data=d, bins=20, fill=MUTED, color="white")
     + geom_vline(aes(xintercept="x"), data=act, color=BLUE, size=1.4)
     + facet_grid(x="period") + scale_x_continuous(format=".1%")
     + labs(title=f"Placebo: Top 20 committed return with {N_PLACEBO} random clusterings (grey) vs AP clusters (blue)",
            subtitle="same number and sizes of clusters in every window; only the assignment of coins is random", x="mean monthly return", y="runs")
     + theme_minimal() + ggsize(900, 340))
show(p, "B5_placebo_clusters")

AP clusters (actual) Random clusters: mean Random clusters: 95th pct Share of random >= actual
Portfolio Period                                                                                               
Top 5     IS                    0.58%                 1.70%                     3.08%                      0.88
          OOS                   1.38%                -0.04%                     1.37%                      0.05
Top 20    IS                    1.40%                 1.66%                     2.16%                      0.81
          OOS                   0.35%                -0.63%                     0.24%                      0.01
All       IS                    1.19%                 1.29%                     1.72%                      0.66
          OOS                   0.08%                -0.78%                     0.09%                      0.06

### B6. Sensitivity: number of principal components (2 SD, Top 20, committed)

In [16]:
rows = []
for nc in PCA_SENSITIVITY:
    if nc == N_COMPONENTS:
        coh = COH["Model 2"]
    else:
        coh = []
        for c in COH["Model 2"]:
            lab_ = cluster_window(c["f0"], c["t0"], c["cols"], nc)["labels"]
            coh.append({**c, "pairs": within(c["X"], c["D"], c["cols"], lab_), "nclus": len(set(lab_))})
    R = run(coh, BASE_SD, 1, log=False)
    for lab, sel in [("In-sample", is_m), ("Out-of-sample", oos_m)]:
        x = R["comm"]["Top 20"]; x = x[sel(x.index)]
        rows.append({"Components": nc, "Period": lab, "Avg pairs": np.mean([len(c["pairs"]) for c in coh]),
                     "Mean / month": x.mean(), "t (NW)": tstat(x), "Sharpe": sharpe(x)})
display(fmt(pd.DataFrame(rows).set_index(["Components", "Period"])))

Avg pairs Mean / month t (NW) Sharpe
Components Period                                            
3          In-sample         24.60        1.42%   1.63   0.66
           Out-of-sample     24.60        0.03%   0.05   0.02
5          In-sample         23.12        1.40%   1.64   0.64
           Out-of-sample     23.12        0.35%   0.84   0.35
10         In-sample         20.44        1.25%   1.84   0.59
           Out-of-sample     20.44       -0.40%  -0.66  -0.39

# Part C – Model 3: entry-threshold grid (both models)
The grid 1.0 / 1.5 / 2.0 / 2.5 / 3.0 SD is run on **both** Model 1 and Model 2, so the effect of the threshold can be separated from the effect of clustering. Each cell is reported gross and net of 10 bp per leg. The Model 3 threshold is picked by the pre-registered rule (highest in-sample Sharpe, Top 20 committed, net of 10 bp) and scored once out-of-sample.

In [17]:
TGRID = {}
for m in ["Model 1", "Model 2"]:
    for sd in THRESHOLDS:
        for bp in (0, SELECT_COST_BP):
            TGRID[(m, sd, bp)] = run(COH[m], sd, 1, bp, log=(bp == 0))
rows = []
for m in ["Model 1", "Model 2"]:
    for sd in THRESHOLDS:
        L = TGRID[(m, sd, 0)]["log"]; L20 = L[(L["rank"] <= 20) & L.complete]
        row = {"Model": m, "Entry SD": sd, "Share of Top 20 pairs that open": (L20.round_trips > 0).mean(),
               "Round trips per opened pair": L20[L20.round_trips > 0].round_trips.mean(),
               "Pairs wiped out": TGRID[(m, sd, 0)]["busts"]}
        for bp in (0, SELECT_COST_BP):
            x = TGRID[(m, sd, bp)]["comm"]["Top 20"]
            for lab, sel in [("IS", is_m), ("OOS", oos_m)]:
                y = x[sel(x.index)]
                row[f"{lab} mean ({bp} bp)"] = y.mean(); row[f"{lab} t ({bp} bp)"] = tstat(y); row[f"{lab} Sharpe ({bp} bp)"] = sharpe(y)
        rows.append(row)
grid = pd.DataFrame(rows).set_index(["Model", "Entry SD"])
pct_cols = tuple(c for c in grid.columns if "mean" in c) + ("Share of Top 20 pairs that open",)
display(fmt(grid, pct=pct_cols))

# ---- pre-registered selection: in-sample only ----
SEL = {}
for m in ["Model 1", "Model 2"]:
    sub = grid.loc[m, f"IS Sharpe ({SELECT_COST_BP} bp)"]
    SEL[m] = float(sub.idxmax())
    print(f"{m}: threshold with the highest in-sample Sharpe net of {SELECT_COST_BP} bp = {SEL[m]} SD "
          f"(IS Sharpe {sub.max():.2f}; at 2 SD {sub.loc[BASE_SD]:.2f})")
SD3 = SEL["Model 2"]
print(f"\nModel 3 = Model 2 clustering with entry threshold {SD3} SD (chosen on in-sample data only)")
print(f"Multiple testing: {len(THRESHOLDS)} thresholds per model; a Bonferroni one-sided 5% bar is t > 2.33, not 1.65.")

Share of Top 20 pairs that open Round trips per opened pair Pairs wiped out IS mean (0 bp) IS t (0 bp) IS Sharpe (0 bp) OOS mean (0 bp) OOS t (0 bp) OOS Sharpe (0 bp) IS mean (10 bp) IS t (10 bp)  \
Model   Entry SD                                                                                                                                                                                                       
Model 1 1.0                               95.29%                        2.13             420          1.02%        0.94             0.52          -0.60%        -0.84             -0.31           0.82%         0.77   
        1.5                               86.45%                        1.82             369          0.88%        0.98             0.48          -0.05%        -0.07             -0.03           0.74%         0.83   
        2.0                               78.43%                        1.66             312          1.59%        1.59             0.79           0.08%         0.19              0.06           1.48%         1.51   
        2.5                               70.35%                        1.56             265          1.64%        1.76             0.85          -0.12%        -0.27             -0.10           1.54%         1.70   
        3.0                               63.95%                        1.48             242          1.66%        1.65             0.84           0.00%         0.01              0.00           1.58%         1.61   
Model 2 1.0                               94.32%                        1.90             318          1.07%        1.11             0.46           0.14%         0.25              0.10           0.85%         0.93   
        1.5                               86.66%                        1.66             288          1.21%        1.35             0.52           0.36%         0.67              0.27           1.08%         1.23   
        2.0                               76.71%                        1.55             249          1.40%        1.64             0.64           0.35%         0.84              0.35           1.29%         1.56   
        2.5                               67.23%                        1.47             211          1.40%        1.99             0.70           0.31%         0.94              0.33           1.31%         1.93   
        3.0                               58.81%                        1.43             177          1.59%        2.18             0.82           0.49%         1.23              0.49           1.52%         2.15   

                 IS Sharpe (10 bp) OOS mean (10 bp) OOS t (10 bp) OOS Sharpe (10 bp)  
Model   Entry SD                                                                      
Model 1 1.0                   0.43           -0.73%         -1.03              -0.38  
        1.5                   0.41           -0.15%         -0.21              -0.08  
        2.0                   0.74            0.00%          0.01               0.00  
        2.5                   0.81           -0.21%         -0.49              -0.17  
        3.0                   0.81           -0.05%         -0.12              -0.04  
Model 2 1.0                   0.38            0.02%          0.03               0.01  
        1.5                   0.47            0.27%          0.50               0.20  
        2.0                   0.59            0.28%          0.67               0.28  
        2.5                   0.66            0.25%          0.77               0.27  
        3.0                   0.79            0.44%          1.11               0.45

Model 1: threshold with the highest in-sample Sharpe net of 10 bp = 3.0 SD (IS Sharpe 0.81; at 2 SD 0.74)
Model 2: threshold with the highest in-sample Sharpe net of 10 bp = 3.0 SD (IS Sharpe 0.79; at 2 SD 0.59)

Model 3 = Model 2 clustering with entry threshold 3.0 SD (chosen on in-sample data only)
Multiple testing: 5 thresholds per model; a Bonferroni one-sided 5% bar is t > 2.33, not 1.65.


In [18]:
hm = []
for m in ["Model 1", "Model 2"]:
    for sd in THRESHOLDS:
        for lab in ["IS", "OOS"]:
            hm.append({"model": m, "Entry SD": f"{sd:.1f}", "period": lab, "Sharpe": grid.loc[(m, sd), f"{lab} Sharpe ({SELECT_COST_BP} bp)"]})
hm = pd.DataFrame(hm)
p = (ggplot(hm, aes("Entry SD", "period", fill="Sharpe")) + geom_tile(color="white")
     + geom_text(aes(label="Sharpe"), label_format=".2f", size=6, color=INK)
     + scale_fill_gradient2(low=RED, mid="#f0efec", high=BLUE, midpoint=0)
     + facet_grid(y="model")
     + labs(title=f"Threshold grid: Sharpe ratio of Top 20, committed capital, net of {SELECT_COST_BP} bp per leg",
            subtitle="the Model 3 threshold is picked from the IS row only", x="entry threshold (formation SD)", y="")
     + theme_minimal() + ggsize(900, 380))
show(p, "C_threshold_grid")

ln = []
for m in ["Model 1", "Model 2"]:
    for sd in THRESHOLDS:
        for lab in ["IS", "OOS"]:
            ln.append({"model": m, "Entry SD": sd, "period": lab, "mean": grid.loc[(m, sd), f"{lab} mean ({SELECT_COST_BP} bp)"]})
ln = pd.DataFrame(ln)
p = (ggplot(ln, aes("Entry SD", "mean", color="model")) + geom_hline(yintercept=0, color=MUTED)
     + geom_line(size=1.2) + geom_point(size=3)
     + facet_grid(x="period") + scale_color_manual(values=[MUTED, BLUE], breaks=["Model 1", "Model 2"])
     + scale_y_continuous(format=".1%")
     + labs(title=f"Mean monthly return by entry threshold, Top 20 committed, net of {SELECT_COST_BP} bp",
            subtitle="a real effect should show a smooth pattern across neighbouring thresholds, not a single spike",
            x="entry threshold (formation SD)", y="", color="")
     + theme_minimal() + theme(legend_position="top") + ggsize(900, 360))
show(p, "C_threshold_lines")

### C1. Model 3 vs Model 2 at 2 SD (test of H2, net of 10 bp)

In [19]:
M3 = TGRID[("Model 2", SD3, SELECT_COST_BP)]; M2n = TGRID[("Model 2", BASE_SD, SELECT_COST_BP)]
RES[("Model 3", 1)] = TGRID[("Model 2", SD3, 0)]
rows = []
for n in PORTS:
    for lab, sel in [("In-sample", is_m), ("Out-of-sample", oos_m)]:
        d = (M3["comm"][n] - M2n["comm"][n]).dropna(); d = d[sel(d.index)]
        rows.append({"Portfolio": n, "Period": lab, f"Model 3 ({SD3} SD) mean": M3["comm"][n][sel(M3["comm"].index)].mean(),
                     "Model 2 (2 SD) mean": M2n["comm"][n][sel(M2n["comm"].index)].mean(),
                     "Difference / month": d.mean(), "t (NW) of difference": tstat(d)})
h2_tab = pd.DataFrame(rows).set_index(["Portfolio", "Period"])
display(fmt(h2_tab, pct=(f"Model 3 ({SD3} SD) mean", "Model 2 (2 SD) mean", "Difference / month")))
print("Model 3 full results, gross of fees:"); display(period_table(RES[("Model 3", 1)]))

Model 3 (3.0 SD) mean Model 2 (2 SD) mean Difference / month t (NW) of difference
Portfolio Period                                                                                         
Top 5     In-sample                     1.63%               0.46%              1.17%                 0.92
          Out-of-sample                 2.05%               1.28%              0.77%                 2.31
Top 20    In-sample                     1.52%               1.29%              0.22%                 0.55
          Out-of-sample                 0.44%               0.28%              0.17%                 0.65
All       In-sample                     1.19%               1.08%              0.11%                 0.30
          Out-of-sample                 0.22%               0.00%              0.21%                 0.76

Model 3 full results, gross of fees:


Dates Months Mean / month (committed) t (NW) committed Sharpe (committed) Mean / month (fully invested) t (NW) fully Sharpe (fully) Max drawdown Months < 0
Portfolio Period                                                                                                                                                                                  
Top 5     In-sample      Mar 2019 – Jul 2024     65                    1.72%             1.74               0.42                         2.81%         2.44           0.65      -67.51%     29.23%
          Out-of-sample  Aug 2024 – Sep 2026     26                    2.13%             2.25               0.91                         3.50%         2.28           1.02      -14.88%     34.62%
          Full           Mar 2019 – Sep 2026     91                    1.83%             2.44               0.50                         3.00%         3.24           0.74      -67.51%     30.77%
Top 20    In-sample      Mar 2019 – Jul 2024     65                    1.59%             2.18               0.82                         2.41%         3.05           1.02      -16.56%     29.23%
          Out-of-sample  Aug 2024 – Sep 2026     26                    0.49%             1.23               0.49                         1.45%         1.28           0.55      -10.50%     42.31%
          Full           Mar 2019 – Sep 2026     91                    1.28%             2.30               0.74                         2.14%         3.18           0.88      -16.56%     32.97%
All       In-sample      Mar 2019 – Jul 2024     65                    1.27%             1.65               0.70                         1.85%         2.04           0.85      -12.57%     32.31%
          Out-of-sample  Aug 2024 – Sep 2026     26                    0.27%             0.87               0.29                         0.47%         0.67           0.21      -10.43%     46.15%
          Full           Mar 2019 – Sep 2026     91                    0.98%             1.72               0.61                         1.46%         2.05           0.66      -12.57%     36.26%

# Part D – Falsification tests shared by all models
### D1. Transaction costs and break-even fee (test of H4)
Fees in bp per leg per execution (a round trip pays 4 legs). The **break-even fee** is where the mean monthly return of the Top 20 (committed) reaches zero, interpolated on the grid. Gatev's implied cost (section 3.3) is shown for comparison; in crypto it mostly measures one-day price moves, not spreads.

In [20]:
SPECS = {"Model 1": ("Model 1", BASE_SD), "Model 2": ("Model 2", BASE_SD), "Model 3": ("Model 2", SD3)}
FEE = {}
for name, (m, sd) in SPECS.items():
    for bp in FEE_GRID_BP:
        FEE[(name, bp)] = TGRID.get((m, sd, bp)) or run(COH[m], sd, 1, bp, log=False)

rows, curve = [], []
for name in SPECS:
    for lab, sel in [("IS", is_m), ("OOS", oos_m)]:
        means = []
        for bp in FEE_GRID_BP:
            x = FEE[(name, bp)]["comm"]["Top 20"]; x = x[sel(x.index)]
            means.append(x.mean()); curve.append({"model": name, "period": lab, "fee": bp, "mean": x.mean(), "t": tstat(x)})
        means = np.array(means)
        if means[0] <= 0: be = 0.0
        elif (means > 0).all(): be = np.inf
        else:
            j = int(np.argmax(means <= 0)); be = np.interp(0, [means[j], means[j - 1]], [FEE_GRID_BP[j], FEE_GRID_BP[j - 1]])
        x10 = FEE[(name, SELECT_COST_BP)]["comm"]["Top 20"]; x10 = x10[sel(x10.index)]
        rows.append({"Model": name, "Period": lab, "Mean gross": means[0], f"Mean at {SELECT_COST_BP} bp": x10.mean(),
                     f"t at {SELECT_COST_BP} bp": tstat(x10), "Break-even fee (bp per leg)": be})
cost_tab = pd.DataFrame(rows).set_index(["Model", "Period"])
_ct = fmt(cost_tab, pct=("Mean gross", f"Mean at {SELECT_COST_BP} bp"))
_ct["Break-even fee (bp per leg)"] = [f"> {FEE_GRID_BP[-1]}" if np.isinf(v) else f"{v:.0f}" for v in cost_tab["Break-even fee (bp per leg)"]]
display(_ct)

imp = {}
for m in ["Model 1", "Model 2"]:
    drop6 = (RES[(m, 0)]["full"]["Top 20"].mean() - RES[(m, 1)]["full"]["Top 20"].mean()) * 6
    rt = table2(RES[(m, 0)]).loc["Average number of round-trip trades per pair", "Top 20"]
    imp[m] = {"Drop from waiting a day, per 6 months (bp)": drop6 * 1e4, "Round trips per pair": rt,
              "Implied cost per round trip (bp)": drop6 * 1e4 / rt}
print("Gatev implied cost (Top 20, 2 SD):"); display(pd.DataFrame(imp).round(1))

cv = pd.DataFrame(curve)
p = (ggplot(cv, aes("fee", "mean", color="model")) + geom_hline(yintercept=0, color=MUTED)
     + geom_line(size=1.2) + geom_point(size=2.5) + facet_grid(x="period")
     + scale_color_manual(values=[MODEL_COL[k] for k in SPECS], breaks=list(SPECS))
     + scale_y_continuous(format=".1%")
     + labs(title="Mean monthly return of Top 20 (committed) as fees rise", subtitle="where a line crosses zero is its break-even fee",
            x="fee, bp per leg per execution", y="", color="")
     + theme_minimal() + theme(legend_position="top") + ggsize(900, 360))
show(p, "D1_costs")

Mean gross Mean at 10 bp t at 10 bp Break-even fee (bp per leg)
Model   Period                                                                
Model 1 IS          1.59%         1.48%       1.51                       > 100
        OOS         0.08%         0.00%       0.01                          11
Model 2 IS          1.40%         1.29%       1.56                       > 100
        OOS         0.35%         0.28%       0.67                          48
Model 3 IS          1.59%         1.52%       2.15                       > 100
        OOS         0.49%         0.44%       1.11                       > 100

Gatev implied cost (Top 20, 2 SD):


,Model 1,Model 2
"Drop from waiting a day, per 6 months (bp)",407.8,123.6
Round trips per pair,1.7,1.6
Implied cost per round trip (bp),244.5,79.3


### D2. Random-pairs bootstrap (Gatev section 3.6; test of H3)
For every actual trade, both coins are replaced by random coins from the same tercile of prior 30-day return on the opening day, keeping the opening and closing dates. 200 runs per model.

In [21]:
def bootstrap(cohorts, sd, n_runs, seed):
    rng = np.random.default_rng(seed); groups = {}
    for c in cohorts:
        elig = np.where(TRADEABLE[c["f0"]:c["t0"]].all(axis=0))[0]
        for r, p in enumerate(c["pairs"]):
            for i, (o, cl_, lg, sh) in enumerate(trades_for(c, p, sd, 1)):
                d0, d1 = c["t0"] + o, c["t0"] + cl_
                cand = elig[TRADEABLE[d0:d1 + 1, elig].all(axis=0) & np.isfinite(P[d0 - 30, elig])]
                terc = pd.qcut(pd.Series(P[d0, cand] / P[d0 - 30, cand] - 1).rank(method="first"), 3, labels=False).to_numpy()
                g = dict(zip(cand, terc))
                groups[(c["k"], r, i)] = (cand[terc == g[lg]], cand[terc == g[sh]])
    out = []
    for _ in range(n_runs):
        cmap = {}
        for c in cohorts:
            for r, p in enumerate(c["pairs"]):
                new = []
                for i, (o, cl_, lg, sh) in enumerate(trades_for(c, p, sd, 1)):
                    gl, gs = groups[(c["k"], r, i)]; L_ = rng.choice(gl); S_ = rng.choice(gs)
                    while S_ == L_ and len(gs) > 1: S_ = rng.choice(gs)
                    new.append((o, cl_, L_, S_))
                cmap[(c["k"], r)] = new
        R = run(cohorts, sd, 1, coin_map=cmap, log=False)
        out.append({(n, lab): R["comm"][n][sel(R["comm"].index)].mean() for n in PORTS for lab, sel in [("IS", is_m), ("OOS", oos_m)]})
    return pd.DataFrame(out)

BOOT = {name: bootstrap(COH[m], sd, N_BOOT, SEED + i) for i, (name, (m, sd)) in enumerate(SPECS.items())}
rows = []
for name in SPECS:
    for n in PORTS:
        for lab, sel in [("IS", is_m), ("OOS", oos_m)]:
            act = RES[(name, 1)]["comm"][n]; act = act[sel(act.index)].mean(); sim = BOOT[name][(n, lab)]
            rows.append({"Model": name, "Portfolio": n, "Period": lab, "Actual": act, "Random: mean": sim.mean(),
                         "Random: SD": sim.std(), "Share of random >= actual": (sim >= act).mean()})
boot_tab = pd.DataFrame(rows).set_index(["Model", "Portfolio", "Period"])
display(fmt(boot_tab, pct=("Actual", "Random: mean", "Random: SD")))

d = pd.concat([pd.DataFrame({"mean monthly return": BOOT[nm][("Top 20", lab)], "model": nm, "period": lab})
               for nm in SPECS for lab in ["IS", "OOS"]])
act = pd.DataFrame([{"model": nm, "period": lab, "x": boot_tab.loc[(nm, "Top 20", lab), "Actual"]} for nm in SPECS for lab in ["IS", "OOS"]])
p = (ggplot() + geom_histogram(aes("mean monthly return"), data=d, bins=20, fill=MUTED, color="white")
     + geom_vline(aes(xintercept="x"), data=act, color=BLUE, size=1.3)
     + facet_grid(x="period", y="model") + scale_x_continuous(format=".1%")
     + labs(title=f"Random-pairs bootstrap, Top 20 committed: {N_BOOT} random portfolios (grey) vs actual (blue)", x="mean monthly return", y="runs")
     + theme_minimal() + ggsize(900, 560))
show(p, "D2_bootstrap")

Actual Random: mean Random: SD Share of random >= actual
Model   Portfolio Period                                                          
Model 1 Top 5     IS       1.46%       -0.16%      0.74%                      0.01
                  OOS      1.25%       -0.46%      1.16%                      0.07
        Top 20    IS       1.59%       -0.12%      0.38%                      0.00
                  OOS      0.08%       -0.55%      0.50%                      0.10
        All       IS       0.96%       -0.12%      0.32%                      0.00
                  OOS     -0.13%       -0.49%      0.47%                      0.26
Model 2 Top 5     IS       0.58%       -0.18%      0.88%                      0.18
                  OOS      1.38%       -0.12%      1.40%                      0.14
        Top 20    IS       1.40%        0.02%      0.42%                      0.00
                  OOS      0.35%       -0.40%      0.57%                      0.08
        All       IS       1.19%        0.02%      0.37%                      0.00
                  OOS      0.08%       -0.42%      0.56%                      0.20
Model 3 Top 5     IS       1.72%       -0.13%      0.82%                      0.02
                  OOS      2.13%        0.27%      1.16%                      0.08
        Top 20    IS       1.59%       -0.03%      0.34%                      0.00
                  OOS      0.49%       -0.10%      0.42%                      0.11
        All       IS       1.27%        0.02%      0.31%                      0.00
                  OOS      0.27%       -0.11%      0.42%                      0.20

### D3. Sub-periods (Top 20, committed, gross)

In [22]:
rows = []
for name in SPECS:
    x = RES[(name, 1)]["comm"]["Top 20"]
    for lab, a, b in SUBPERIODS:
        y = x.loc[pd.Period(a, "M"):pd.Period(b, "M")]
        rows.append({"Model": name, "Sub-period": f"{lab} ({a} to {b})", **summ(y)})
display(fmt(pd.DataFrame(rows).set_index(["Model", "Sub-period"])))

Months Mean / month t (NW) Sharpe Max drawdown Months < 0
Model   Sub-period                                                                                    
Model 1 2019–2020 (2019-03 to 2020-12)           22        1.43%   1.26   0.93      -12.88%     50.00%
        2021–2022 (2021-01 to 2022-12)           24        3.74%   1.67   1.45       -9.73%     25.00%
        2023 – Jul 2024 (2023-01 to 2024-07)     19       -0.94%  -0.93  -0.63      -33.51%     47.37%
        OOS (2024-08 to 2026-09)                 26        0.08%   0.19   0.06      -18.00%     38.46%
Model 2 2019–2020 (2019-03 to 2020-12)           22        1.47%   1.45   0.98      -15.51%     22.73%
        2021–2022 (2021-01 to 2022-12)           24        2.54%   1.18   0.79      -12.64%     41.67%
        2023 – Jul 2024 (2023-01 to 2024-07)     19       -0.12%  -0.23  -0.12      -16.47%     52.63%
        OOS (2024-08 to 2026-09)                 26        0.35%   0.84   0.35      -13.49%     42.31%
Model 3 2019–2020 (2019-03 to 2020-12)           22        1.11%   2.25   1.09      -10.51%     18.18%
        2021–2022 (2021-01 to 2022-12)           24        2.61%   1.31   0.87       -9.95%     37.50%
        2023 – Jul 2024 (2023-01 to 2024-07)     19        0.87%   2.12   1.16       -5.90%     31.58%
        OOS (2024-08 to 2026-09)                 26        0.49%   1.23   0.49      -10.50%     42.31%

### D4. Risk exposure (Gatev section 3.5, Table 4 – crypto factors)
Monthly Top 20 fully invested returns (gross, one-day wait) on crypto factors from the same coins: **Market** (average coin return), **Size proxy** (low minus high prior-month dollar volume terciles – no market cap in the data), **Momentum** (top minus bottom tercile, months *t*−12 to *t*−2), **Reversal** (top minus bottom tercile of prior-month return). Newey–West t-statistics in brackets.

In [23]:
rd = px.pct_change(fill_method=None); grp = rd.index.to_period("M")
cm = (1 + rd.fillna(0)).groupby(grp).prod() - 1; cm[rd.notna().groupby(grp).sum() == 0] = np.nan
dvol = vlm.groupby(vlm.index.to_period("M")).mean()
def spread(sort_vals, rets):
    s = sort_vals.dropna(); s = s[rets.reindex(s.index).notna()]
    if len(s) < 9: return np.nan
    q = pd.qcut(s.rank(method="first"), 3, labels=False)
    return rets[q[q == 2].index].mean() - rets[q[q == 0].index].mean()
F = {}
for i, m_ in enumerate(cm.index):
    if i < 12: continue
    mom = (1 + cm.iloc[i - 12:i - 1]).prod() - 1
    F[m_] = {"Market": cm.loc[m_].mean(), "Size proxy": -spread(dvol.iloc[i - 1], cm.loc[m_]),
             "Momentum": spread(mom.where(cm.iloc[i - 12:i - 1].notna().all()), cm.loc[m_]), "Reversal": spread(cm.iloc[i - 1], cm.loc[m_])}
F = pd.DataFrame(F).T
def ols_nw(y, X, lags=NW_LAGS):
    X = np.column_stack([np.ones(len(X)), X]); b = np.linalg.lstsq(X, y, rcond=None)[0]; e = y - X @ b; T = len(y)
    S = (X * e[:, None]).T @ (X * e[:, None]) / T
    for l in range(1, lags + 1):
        G = (X[l:] * e[l:, None]).T @ (X[:-l] * e[:-l, None]) / T; S += (1 - l / (lags + 1)) * (G + G.T)
    XX = np.linalg.inv(X.T @ X / T); se = np.sqrt(np.diag(XX @ S @ XX / T))
    return b, b / se, 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
tab = {}
for name in SPECS:
    d = pd.concat([RES[(name, 1)]["full"]["Top 20"].rename("y"), F], axis=1).dropna()
    b, t, r2 = ols_nw(d.y.values, d[F.columns].values)
    tab[name] = {k: f"{bi:.4f} ({ti:.2f})" for k, bi, ti in zip(["Intercept"] + list(F.columns), b, t)} | {"R²": f"{r2:.2f}", "Months": len(d)}
display(pd.DataFrame(tab))

,Model 1,Model 2,Model 3
Intercept,0.0119 (1.96),0.0097 (1.80),0.0174 (3.19)
Market,-0.0067 (-0.21),0.0602 (0.98),0.0794 (1.21)
Size proxy,0.0120 (0.13),0.0138 (0.20),0.0313 (0.42)
Momentum,-0.0714 (-1.22),-0.1286 (-2.16),-0.1769 (-2.26)
Reversal,-0.1723 (-1.92),-0.1009 (-1.61),-0.0731 (-0.75)
R²,0.13,0.15,0.15
Months,91,91,91


### D5. Performance over time

In [24]:
oos_x = OOS0.to_timestamp(); end_x = months[-1].to_timestamp(how="end")
g = []
for name in SPECS:
    for lab, R in [("gross", RES[(name, 1)]), (f"net of {SELECT_COST_BP} bp", FEE[(name, SELECT_COST_BP)])]:
        x = R["comm"]["Top 20"].dropna(); w = (1 + x).cumprod()
        g.append(pd.DataFrame({"date": x.index.to_timestamp(how="end"), "value": w.values, "model": name, "fees": lab}))
g = pd.concat(g)
order = list(SPECS); legend_lab = {"Model 1": "Model 1 (2 SD)", "Model 2": "Model 2 (2 SD)", "Model 3": f"Model 3 ({SD3} SD)"}
g["model"] = g["model"].map(legend_lab)
shade = pd.DataFrame({"xmin": [oos_x], "xmax": [end_x], "ymin": [g.value.min() * 0.95], "ymax": [g.value.max() * 1.03]})
p = (ggplot() + geom_rect(aes(xmin="xmin", xmax="xmax", ymin="ymin", ymax="ymax"), data=shade, fill=GRID, color=GRID, alpha=0.7)
     + geom_hline(yintercept=1, color=MUTED)
     + geom_line(aes("date", "value", color="model"), data=g, size=1.1) + facet_grid(x="fees")
     + scale_color_manual(values=[MODEL_COL[k] for k in order], breaks=[legend_lab[k] for k in order])
     + scale_x_datetime(format="%Y") + scale_y_continuous(format="$.2f")
     + labs(title="Growth of $1, Top 20 committed capital, one-day wait", subtitle="shaded = out-of-sample", x="", y="", color="")
     + theme_minimal() + theme(legend_position="top") + ggsize(1000, 440))
show(p, "D5_growth")

dd = []
for name in SPECS:
    x = RES[(name, 1)]["comm"]["Top 20"].dropna(); w = np.concatenate([[1.0], (1 + x).cumprod().values])
    dd.append(pd.DataFrame({"date": x.index.to_timestamp(how="end"), "drawdown": (w / np.maximum.accumulate(w) - 1)[1:], "model": legend_lab[name]}))
dd = pd.concat(dd)
shade2 = pd.DataFrame({"xmin": [oos_x], "xmax": [end_x], "ymin": [dd.drawdown.min() * 1.05], "ymax": [0.0]})
p = (ggplot() + geom_rect(aes(xmin="xmin", xmax="xmax", ymin="ymin", ymax="ymax"), data=shade2, fill=GRID, color=GRID, alpha=0.7)
     + geom_line(aes("date", "drawdown", color="model"), data=dd, size=1.1)
     + scale_color_manual(values=[MODEL_COL[k] for k in order], breaks=[legend_lab[k] for k in order])
     + scale_x_datetime(format="%Y") + scale_y_continuous(format=".0%")
     + labs(title="Drawdown from previous peak, Top 20 committed, gross", x="", y="", color="")
     + theme_minimal() + theme(legend_position="top") + ggsize(1000, 320))
show(p, "D5_drawdown")

# Part E – Verdict on the pre-registered hypotheses
Filled in automatically from the tests above, with the decision rules fixed at the top of the notebook.

In [25]:
def verdict(ok): return "supported" if ok else "NOT supported (falsified at this level)"
h1_diff = cmp12.loc[("Top 20", "Out-of-sample")]
h1_pl = placebo_tab.loc[("Top 20", "OOS")]
h2 = h2_tab.loc[("Top 20", "Out-of-sample")]
rows = [
    {"Hypothesis": "H1a Clustering beats Gatev (Model 2 − Model 1, OOS, Top 20)",
     "Statistic": f"{h1_diff['Difference / month']:.2%} per month, t = {h1_diff['t (NW) of difference']:.2f}",
     "Rule": f"t > {T_CRIT}", "Verdict": verdict(h1_diff["t (NW) of difference"] > T_CRIT)},
    {"Hypothesis": "H1b AP clusters beat random clusters (OOS, Top 20)",
     "Statistic": f"{h1_pl['Share of random >= actual']:.0%} of {N_PLACEBO} placebo runs do as well",
     "Rule": "< 5%", "Verdict": verdict(h1_pl["Share of random >= actual"] < 0.05)},
    {"Hypothesis": "H1b check: the same test in-sample (not the pre-registered rule, shown for transparency)",
     "Statistic": f"{placebo_tab.loc[('Top 20', 'IS'), 'Share of random >= actual']:.0%} of {N_PLACEBO} placebo runs do as well",
     "Rule": "< 5%", "Verdict": verdict(placebo_tab.loc[("Top 20", "IS"), "Share of random >= actual"] < 0.05)},
    {"Hypothesis": f"H2 IS-chosen threshold ({SD3} SD) beats 2 SD (Model 3 − Model 2, OOS, net of {SELECT_COST_BP} bp)",
     "Statistic": ("same threshold chosen – no difference to test" if SD3 == BASE_SD else
                   f"{h2['Difference / month']:.2%} per month, t = {h2['t (NW) of difference']:.2f}"),
     "Rule": f"t > {T_CRIT}", "Verdict": verdict(SD3 != BASE_SD and h2["t (NW) of difference"] > T_CRIT)},
]
for name in SPECS:
    b = boot_tab.loc[(name, "Top 20", "OOS")]
    rows.append({"Hypothesis": f"H3 {name} pairs beat random pairs (OOS, Top 20)",
                 "Statistic": f"{b['Share of random >= actual']:.0%} of {N_BOOT} random runs do as well",
                 "Rule": "< 5%", "Verdict": verdict(b["Share of random >= actual"] < 0.05)})
for name in SPECS:
    c = cost_tab.loc[(name, "OOS")]
    rows.append({"Hypothesis": f"H4 {name} profitable after {SELECT_COST_BP} bp (OOS, Top 20)",
                 "Statistic": f"{c[f'Mean at {SELECT_COST_BP} bp']:.2%} per month, t = {c[f't at {SELECT_COST_BP} bp']:.2f}; "
                              + ("break-even > %d bp" % FEE_GRID_BP[-1] if np.isinf(c['Break-even fee (bp per leg)']) else f"break-even {c['Break-even fee (bp per leg)']:.0f} bp"),
                 "Rule": f"t > {T_CRIT}", "Verdict": verdict(c[f"t at {SELECT_COST_BP} bp"] > T_CRIT)})
display(pd.DataFrame(rows).set_index("Hypothesis"))

,Statistic,Rule,Verdict
Hypothesis,,,
"H1a Clustering beats Gatev (Model 2 − Model 1, OOS, Top 20)","0.27% per month, t = 0.52",t > 1.65,NOT supported (falsified at this level)
"H1b AP clusters beat random clusters (OOS, Top 20)",1% of 100 placebo runs do as well,< 5%,supported
"H1b check: the same test in-sample (not the pre-registered rule, shown for transparency)",81% of 100 placebo runs do as well,< 5%,NOT supported (falsified at this level)
"H2 IS-chosen threshold (3.0 SD) beats 2 SD (Model 3 − Model 2, OOS, net of 10 bp)","0.17% per month, t = 0.65",t > 1.65,NOT supported (falsified at this level)
"H3 Model 1 pairs beat random pairs (OOS, Top 20)",10% of 200 random runs do as well,< 5%,NOT supported (falsified at this level)
"H3 Model 2 pairs beat random pairs (OOS, Top 20)",8% of 200 random runs do as well,< 5%,NOT supported (falsified at this level)
"H3 Model 3 pairs beat random pairs (OOS, Top 20)",11% of 200 random runs do as well,< 5%,NOT supported (falsified at this level)
"H4 Model 1 profitable after 10 bp (OOS, Top 20)","0.00% per month, t = 0.01; break-even 11 bp",t > 1.65,NOT supported (falsified at this level)
"H4 Model 2 profitable after 10 bp (OOS, Top 20)","0.28% per month, t = 0.67; break-even 48 bp",t > 1.65,NOT supported (falsified at this level)


## Notes and limitations
- **One engine:** all three models share the same data, screen, trading code and statistics; Models differ only where the table at the top says.
- **Pairs 101–120:** with ~35 coins best-partner matching gives ~20–30 pairs per window, so Gatev's 101–120 portfolio cannot exist. "All" is every best-partner pair. When Model 2 has fewer than 20 within-cluster pairs, "Top 20" holds all of them.
- **Unbounded short losses:** the \$1 legs are never rebalanced (Gatev's buy-and-hold). A short leg can lose more than the pair's capital, especially on gap days (e.g. BNB in February 2021); pairs are closed when their capital reaches zero, and the loss on that day is booked in full.
- **Survivorship:** only coins with prices up to 2026 are in the file, which flatters returns. Several coins have long zero-volume stretches; the screen removes them from formation and the delisting rule closes positions on their first no-trade day.
- **Costs:** the fee grid covers exchange fees, spread and slippage together; borrowing/funding costs on shorts are not modelled, so true break-even fees are lower than shown.
- **Multiple testing:** the threshold grid adds 5 trials per model; the H2 test uses one pre-registered choice, and the Bonferroni bar (t > 2.33) is printed for reference.
- **OOS not pristine for Models 1 and 2:** their 2 SD OOS results had been seen before this notebook; the grid, selection rule, placebo and verdict rules were fixed before running it.
- **Factors:** no market cap in the data, so size uses dollar volume; no S&P 500 or T-bill series is used; returns are self-financing excess returns, as in Gatev.